# Combined ROC Curve â€” All Models

Self-contained notebook. Downloads data from GitHub, trains 8 models on
the same split, exports ROC data + a combined chart.

| Family | Models |
|--------|--------|
| CatBoost | Core Only Â· Core+Labs Â· Full Model |
| sklearn ANN (MLPClassifier) | best feature set (full) |
| PyTorch Tabular | best feature set (full) |
| Logistic Regression | full feature set |
| SVM (RBF) | full feature set |
| Decision Tree | full feature set |

In [ ]:
!pip install -q catboost

In [ ]:
import os, random, json, math, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    average_precision_score, brier_score_loss,
    confusion_matrix, accuracy_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

SEED = 42
TARGET_RECALL = 0.85
TARGET_CAT = 'has_diabetes'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Imports OK')
print('CUDA:', torch.cuda.is_available())

## 1. Config

In [ ]:
GH_URL = 'https://github.com/tjohns94/cds492-3c-project/raw/refs/heads/main/data/health_2024.CSV'
DATA_ENCODING = 'cp949'

COLUMN_NAMES_INIT = [
    'year', 'subscriber_id', 'city_code',
    'sex_code', 'age_group_code', 'height',
    'weight', 'waist_circumference', 'vision_left',
    'vision_right', 'hearing_left', 'hearing_right',
    'systolic_bp', 'diastolic_bp', 'fpg',
    'total_cholesterol', 'triglycerides', 'hdl_cholesterol',
    'ldl_cholesterol', 'hemoglobin', 'urine_protein',
    'serum_creatinine', 'serum_got_ast', 'serum_gpt_alt',
    'gamma_gtp', 'smoking_status', 'alcohol_consumption',
    'oral_exam', 'caries_presence', 'missing_teeth_presence',
    'tooth_wear_presence', 'wisdom_teeth_abnormality', 'plaque_presence',
]
COLUMN_NAMES_TO_DROP = [
    'year', 'subscriber_id', 'missing_teeth_presence',
    'tooth_wear_presence', 'wisdom_teeth_abnormality',
]
COLUMN_TYPES = {
    'sex_code': ('category','int'), 'city_code': ('category','int'),
    'hearing_left': ('category','int'), 'hearing_right': ('category','int'),
    'alcohol_consumption': ('category','int'), 'oral_exam': ('category','int'),
    'caries_presence': ('category','int'), 'plaque_presence': ('category','int'),
    'age_group_code': ('category','int'), 'smoking_status': ('category','int'),
    'urine_protein': ('category','int'),
    'height': ('numeric','int'), 'weight': ('numeric','int'),
    'waist_circumference': ('numeric','float'),
    'systolic_bp': ('numeric','int'), 'diastolic_bp': ('numeric','int'),
    'vision_left': ('numeric','float'), 'vision_right': ('numeric','float'),
    'total_cholesterol': ('numeric','int'), 'triglycerides': ('numeric','int'),
    'hdl_cholesterol': ('numeric','int'), 'ldl_cholesterol': ('numeric','int'),
    'hemoglobin': ('numeric','float'), 'serum_creatinine': ('numeric','float'),
    'serum_got_ast': ('numeric','int'), 'serum_gpt_alt': ('numeric','int'),
    'gamma_gtp': ('numeric','int'), 'fpg': ('numeric','int'),
}
COLUMN_DOMAINS = {
    'sex_code': {1,2}, 'city_code': {11,26,27,28,29,30,31,36,41,42,43,44,45,46,47},
    'hearing_left': {1,2}, 'hearing_right': {1,2},
    'alcohol_consumption': {0,1}, 'oral_exam': {0,1},
    'caries_presence': {0,1}, 'plaque_presence': {0,1},
    'age_group_code': set(range(5,19)), 'smoking_status': {1,2,3},
    'urine_protein': set(range(1,7)),
    'height': [100,230], 'weight': [20,250], 'waist_circumference': [30,250],
    'systolic_bp': [60,300], 'diastolic_bp': [30,200],
    'vision_left': [0.1,2.5], 'vision_right': [0.1,2.5],
    'total_cholesterol': [50,1000], 'triglycerides': [10,5000],
    'hdl_cholesterol': [5,200], 'ldl_cholesterol': [5,700],
    'hemoglobin': [3,25], 'serum_creatinine': [0.1,30],
    'serum_got_ast': [1,5000], 'serum_gpt_alt': [1,5000],
    'gamma_gtp': [1,5000], 'fpg': [30,1000],
}

FEATURES_NUMERIC = [c for c,(t,_) in COLUMN_TYPES.items() if t=='numeric' and c!='fpg']
FEATURES_CATEGORICAL = [c for c,(t,_) in COLUMN_TYPES.items() if t=='category']
COLUMNS_ALL = FEATURES_NUMERIC + FEATURES_CATEGORICAL + ['fpg']

CORE_NUMERIC = [
    'age_years_mid','height','weight','waist_circumference',
    'bmi','waist_to_height','wwi',
    'systolic_bp','diastolic_bp','pulse_pressure','mean_arterial_pressure',
    'urine_protein',
]
CORE_NOMINAL = ['sex_code','smoking_status','alcohol_consumption']

NONLIPID_LAB = [
    'hemoglobin','serum_creatinine',
    'log_ast','log_alt','log_ggt','ast_alt_ratio',
]
NONLIPID_FLAGS = [
    'missing_hemoglobin','missing_serum_creatinine',
    'missing_serum_got_ast','missing_serum_gpt_alt','missing_gamma_gtp',
]

LIPID_FEATURES = [
    'total_cholesterol','hdl_cholesterol','ldl_cholesterol',
    'log_triglycerides','non_hdl_cholesterol',
    'tg_hdl_ratio','tc_hdl_ratio','ldl_hdl_ratio',
    'has_complete_lipid_panel','n_lipids_available',
]
LIPID_FLAGS = [
    'missing_total_cholesterol','missing_triglycerides',
    'missing_hdl_cholesterol','missing_ldl_cholesterol',
]

FEATURE_SETS = {
    'core_screening': {
        'features': CORE_NUMERIC + CORE_NOMINAL,
        'nominal_features': CORE_NOMINAL,
    },
    'core_plus_labs': {
        'features': CORE_NUMERIC + CORE_NOMINAL + NONLIPID_LAB + NONLIPID_FLAGS,
        'nominal_features': CORE_NOMINAL,
    },
    'core_plus_labs_plus_lipids': {
        'features': CORE_NUMERIC + CORE_NOMINAL + NONLIPID_LAB + NONLIPID_FLAGS + LIPID_FEATURES + LIPID_FLAGS,
        'nominal_features': CORE_NOMINAL,
    },
}

print('Config loaded')

## 2. Download & Preprocess

In [ ]:
def download_dataset(url=GH_URL, path='health_2024.csv'):
    if os.path.exists(path):
        print('Dataset already downloaded.')
        return path
    print(f'Downloading from {url} ...')
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    with open(path, 'wb') as f:
        f.write(r.content)
    print(f'Saved ({len(r.content)/1e6:.1f} MB)')
    return path

def load_and_clean(path):
    df = pd.read_csv(path, encoding=DATA_ENCODING)
    df.columns = COLUMN_NAMES_INIT
    df = df.drop(columns=COLUMN_NAMES_TO_DROP)
    df = df[COLUMNS_ALL]
    for col, (_, dtype) in COLUMN_TYPES.items():
        if col not in df.columns: continue
        if dtype == 'int':
            df[col] = pd.to_numeric(df[col], errors='coerce').round().astype('Int64')
        elif dtype == 'float':
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Float64')
        domain = COLUMN_DOMAINS[col]
        if isinstance(domain, set):
            df.loc[~df[col].isin(domain), col] = pd.NA
        else:
            lo, hi = domain
            df.loc[(df[col] < lo) | (df[col] > hi), col] = pd.NA
    df['blind_left']  = (df['vision_left'] == 9.9).astype('Int64')
    df['blind_right'] = (df['vision_right'] == 9.9).astype('Int64')
    df.loc[df['blind_left'] == 1, 'vision_left'] = pd.NA
    df.loc[df['blind_right'] == 1, 'vision_right'] = pd.NA
    df['has_diabetes'] = (df['fpg'] >= 126).astype('Int64')
    df['wwi'] = df['waist_circumference'] / np.sqrt(df['weight'])
    df['age_years_mid'] = pd.to_numeric(df['age_group_code'], errors='coerce').astype(float) * 5 - 2.5
    h_m  = pd.to_numeric(df['height'], errors='coerce').astype(float) / 100.0
    h_cm = pd.to_numeric(df['height'], errors='coerce').astype(float)
    w    = pd.to_numeric(df['weight'], errors='coerce').astype(float)
    waist = pd.to_numeric(df['waist_circumference'], errors='coerce').astype(float)
    sys  = pd.to_numeric(df['systolic_bp'], errors='coerce').astype(float)
    dia  = pd.to_numeric(df['diastolic_bp'], errors='coerce').astype(float)
    ast  = pd.to_numeric(df['serum_got_ast'], errors='coerce').astype(float)
    alt  = pd.to_numeric(df['serum_gpt_alt'], errors='coerce').astype(float)
    ggt  = pd.to_numeric(df['gamma_gtp'], errors='coerce').astype(float)
    tc   = pd.to_numeric(df['total_cholesterol'], errors='coerce').astype(float)
    tg   = pd.to_numeric(df['triglycerides'], errors='coerce').astype(float)
    hdl  = pd.to_numeric(df['hdl_cholesterol'], errors='coerce').astype(float)
    ldl  = pd.to_numeric(df['ldl_cholesterol'], errors='coerce').astype(float)
    def _sd(n, d):
        o = n / d; o[(d <= 0) | d.isna()] = np.nan; return o
    df['bmi'] = _sd(w, h_m**2)
    df['waist_to_height'] = _sd(waist, h_cm)
    df['pulse_pressure'] = sys - dia
    df['mean_arterial_pressure'] = (sys + 2*dia) / 3.0
    df['ast_alt_ratio'] = _sd(ast, alt)
    df['log_ast'] = np.log1p(ast)
    df['log_alt'] = np.log1p(alt)
    df['log_ggt'] = np.log1p(ggt)
    df['log_triglycerides'] = np.log1p(tg)
    df['non_hdl_cholesterol'] = tc - hdl
    df['tg_hdl_ratio'] = _sd(tg, hdl)
    df['tc_hdl_ratio'] = _sd(tc, hdl)
    df['ldl_hdl_ratio'] = _sd(ldl, hdl)
    for col in ['hemoglobin','serum_creatinine','serum_got_ast','serum_gpt_alt',
                'gamma_gtp','total_cholesterol','triglycerides','hdl_cholesterol','ldl_cholesterol']:
        df[f'missing_{col}'] = df[col].isna().astype(int)
    lipid_cols = ['total_cholesterol','triglycerides','hdl_cholesterol','ldl_cholesterol']
    df['has_complete_lipid_panel'] = df[lipid_cols].notna().all(axis=1).astype(int)
    df['n_lipids_available'] = df[lipid_cols].notna().sum(axis=1).astype(int)
    return df

data_path = download_dataset()
df = load_and_clean(data_path)
print(f'Dataset: {df.shape}')

## 3. Build Shared Split

In [ ]:
all_features = list(dict.fromkeys(
    CORE_NUMERIC + CORE_NOMINAL + NONLIPID_LAB + NONLIPID_FLAGS + LIPID_FEATURES + LIPID_FLAGS
))
available = [f for f in all_features if f in df.columns]
df_model = df[available + [TARGET_CAT]].copy()
df_model = df_model[df_model[TARGET_CAT].notna()].reset_index(drop=True)

y = df_model[TARGET_CAT].astype(int)
X = df_model[available].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train
)

pos = int(y_train.sum())
neg = int(len(y_train) - pos)
cw_dict = {0: 1.0, 1: neg / max(pos, 1)}
cw_list = [1.0, neg / max(pos, 1)]

print(f'Train: {len(y_train):,}  Valid: {len(y_valid):,}  Test: {len(y_test):,}')
print(f'Positive rate: {y.mean():.3%}')

## 4. Shared Helper Functions

In [ ]:
# --- Calibration ---
def prob_to_logit(p, eps=1e-6):
    p = np.clip(np.asarray(p, float), eps, 1-eps)
    return np.log(p / (1-p))

def fit_calibrator(raw_prob, y_true):
    cal = LogisticRegression(random_state=SEED)
    cal.fit(prob_to_logit(raw_prob).reshape(-1,1), y_true)
    return cal

def apply_calibrator(cal, raw_prob):
    return cal.predict_proba(prob_to_logit(raw_prob).reshape(-1,1))[:, 1]

# --- CatBoost input formatting ---
def prepare_catboost_inputs(Xtr, Xv, Xte, nominal=CORE_NOMINAL):
    frames = []
    for d in [Xtr, Xv, Xte]:
        d = d.copy()
        for col in d.columns:
            if col in nominal:
                d[col] = d[col].astype('string').fillna('Missing')
            else:
                d[col] = pd.to_numeric(d[col], errors='coerce').astype(float)
        frames.append(d)
    cat_idx = [Xtr.columns.get_loc(c) for c in nominal if c in Xtr.columns]
    return frames[0], frames[1], frames[2], cat_idx

# --- ANN preprocessing (median impute + scale + one-hot) ---
def preprocess_ann_inputs(Xtr, Xv, Xte, nominal_features):
    Xtr, Xv, Xte = Xtr.copy(), Xv.copy(), Xte.copy()
    nom = [c for c in nominal_features if c in Xtr.columns]
    num = [c for c in Xtr.columns if c not in nom]
    Xtr_n = Xtr[num].apply(pd.to_numeric, errors='coerce').astype(float)
    Xv_n  = Xv[num].apply(pd.to_numeric, errors='coerce').astype(float)
    Xte_n = Xte[num].apply(pd.to_numeric, errors='coerce').astype(float)
    meds = Xtr_n.median().fillna(0.0)
    Xtr_n, Xv_n, Xte_n = Xtr_n.fillna(meds), Xv_n.fillna(meds), Xte_n.fillna(meds)
    scaler = StandardScaler()
    Xtr_n = pd.DataFrame(scaler.fit_transform(Xtr_n), columns=num, index=Xtr.index)
    Xv_n  = pd.DataFrame(scaler.transform(Xv_n), columns=num, index=Xv.index)
    Xte_n = pd.DataFrame(scaler.transform(Xte_n), columns=num, index=Xte.index)
    def enc(d):
        if len(nom)==0: return pd.DataFrame(index=d.index)
        o = d[nom].copy()
        for c in nom: o[c] = o[c].astype('string').fillna('Missing')
        return pd.get_dummies(o, columns=nom, drop_first=False, dtype=float)
    Xtr_c = enc(Xtr)
    Xv_c  = enc(Xv).reindex(columns=Xtr_c.columns, fill_value=0.0)
    Xte_c = enc(Xte).reindex(columns=Xtr_c.columns, fill_value=0.0)
    return (
        pd.concat([Xtr_n, Xtr_c], axis=1),
        pd.concat([Xv_n, Xv_c], axis=1),
        pd.concat([Xte_n, Xte_c], axis=1),
    )

# --- Rebalance training data ---
def rebalance_training_data(Xtr, ytr, target_pos_rate=0.30, seed=42):
    d = Xtr.copy(); d['_t'] = ytr.to_numpy()
    pos, neg = d[d['_t']==1], d[d['_t']==0]
    if len(pos)==0 or len(neg)==0: return Xtr.copy(), ytr.copy()
    if len(pos)/len(d) >= target_pos_rate: return Xtr.copy(), ytr.copy()
    n_pos = max(int(target_pos_rate * len(neg) / (1-target_pos_rate)), len(pos))
    pos_up = resample(pos, replace=True, n_samples=n_pos, random_state=seed)
    bal = pd.concat([neg, pos_up]).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return bal.drop(columns='_t'), bal['_t'].astype(int)

# --- Torch helpers ---
class TorchTabularDataset(Dataset):
    def __init__(self, x_num, x_cat, y=None):
        self.x_num = torch.tensor(x_num, dtype=torch.float32)
        self.x_cat = torch.tensor(x_cat, dtype=torch.long)
        self.y = None if y is None else torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.x_num)
    def __getitem__(self, i):
        if self.y is None: return self.x_num[i], self.x_cat[i]
        return self.x_num[i], self.x_cat[i], self.y[i]

def embedding_dim_for_cardinality(c):
    if c <= 2: return 1
    return min(32, max(2, int(round(math.sqrt(c)))))

class TorchTabularNet(nn.Module):
    def __init__(self, num_numeric, cat_cards, hidden_dims=(128,64), dropout=0.20):
        super().__init__()
        self.embs = nn.ModuleList()
        emb_out = 0
        for card in cat_cards:
            d = embedding_dim_for_cardinality(card)
            self.embs.append(nn.Embedding(card, d))
            emb_out += d
        layers = []
        prev = num_numeric + emb_out
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.mlp = nn.Sequential(*layers)
    def forward(self, x_num, x_cat):
        parts = [x_num]
        if len(self.embs) > 0:
            parts.append(torch.cat([e(x_cat[:,i]) for i,e in enumerate(self.embs)], dim=1))
        return self.mlp(torch.cat(parts, dim=1)).squeeze(1)

def predict_torch_prob(model, loader, device):
    model.eval()
    probs = []
    with torch.inference_mode():
        for batch in loader:
            xn, xc = batch[0].to(device), batch[1].to(device)
            probs.append(torch.sigmoid(model(xn, xc)).cpu().numpy())
    return np.concatenate(probs)

def prepare_torch_inputs(Xtr, Xv, Xte, nominal_features):
    Xtr, Xv, Xte = Xtr.copy(), Xv.copy(), Xte.copy()
    cats = [c for c in nominal_features if c in Xtr.columns]
    nums = [c for c in Xtr.columns if c not in cats]
    Xtr_n = Xtr[nums].apply(pd.to_numeric, errors='coerce').astype(float)
    Xv_n  = Xv[nums].apply(pd.to_numeric, errors='coerce').astype(float)
    Xte_n = Xte[nums].apply(pd.to_numeric, errors='coerce').astype(float)
    meds = Xtr_n.median().fillna(0.0)
    Xtr_n, Xv_n, Xte_n = Xtr_n.fillna(meds), Xv_n.fillna(meds), Xte_n.fillna(meds)
    sc = StandardScaler()
    Xtr_n = sc.fit_transform(Xtr_n)
    Xv_n  = sc.transform(Xv_n)
    Xte_n = sc.transform(Xte_n)
    for d in [Xtr, Xv, Xte]:
        for c in cats: d[c] = d[c].astype('string').fillna('Missing')
    mappings = {}
    for c in cats:
        vals = Xtr[c].unique().tolist()
        mappings[c] = {v: i+1 for i,v in enumerate(vals)}
    def enc(d):
        if not cats: return np.zeros((len(d),0), dtype=np.int64)
        cols = [d[c].map(mappings[c]).fillna(0).astype(int).to_numpy() for c in cats]
        return np.column_stack(cols).astype(np.int64)
    cards = [max(mappings[c].values(), default=0)+1 for c in cats]
    return {
        'Xtr_n': Xtr_n.astype(np.float32), 'Xv_n': Xv_n.astype(np.float32), 'Xte_n': Xte_n.astype(np.float32),
        'Xtr_c': enc(Xtr), 'Xv_c': enc(Xv), 'Xte_c': enc(Xte),
        'cards': cards,
    }

print('Helpers defined')

## 5. Train CatBoost Ã— 3

In [ ]:
roc_data = {}  # name -> {fpr, tpr, auc}

CB_SETS = {
    'CatBoost (Core Only)':   FEATURE_SETS['core_screening'],
    'CatBoost (Core + Labs)': FEATURE_SETS['core_plus_labs'],
    'CatBoost (Full Model)':  FEATURE_SETS['core_plus_labs_plus_lipids'],
}

for name, spec in CB_SETS.items():
    feats = [f for f in spec['features'] if f in X_train.columns]
    print(f'\n{"="*60}\n{name}  ({len(feats)} features)\n{"="*60}')
    Xtr, Xv, Xte, cat_idx = prepare_catboost_inputs(
        X_train[feats], X_valid[feats], X_test[feats]
    )
    m = CatBoostClassifier(
        loss_function='Logloss', eval_metric='AUC',
        iterations=1500, learning_rate=0.03,
        depth=6, l2_leaf_reg=5.0,
        random_seed=SEED, verbose=500,
        allow_writing_files=False, class_weights=cw_list,
    )
    m.fit(Xtr, y_train, cat_features=cat_idx,
          eval_set=(Xv, y_valid), use_best_model=True, early_stopping_rounds=100)
    rv = m.predict_proba(Xv)[:, 1]
    rt = m.predict_proba(Xte)[:, 1]
    cal = fit_calibrator(rv, y_valid)
    ct = apply_calibrator(cal, rt)
    auc = roc_auc_score(y_test, ct)
    fpr, tpr, _ = roc_curve(y_test, ct)
    roc_data[name] = {'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'auc': round(auc, 3)}
    print(f'  => ROC-AUC = {auc:.4f}')

## 6. Train sklearn ANN (MLPClassifier)

In [ ]:
# Use the full feature set for the ANN
spec = FEATURE_SETS['core_plus_labs_plus_lipids']
feats = [f for f in spec['features'] if f in X_train.columns]
nom = spec['nominal_features']

print(f'Training sklearn ANN on {len(feats)} features...')

Xtr_a, Xv_a, Xte_a = preprocess_ann_inputs(
    X_train[feats], X_valid[feats], X_test[feats], nominal_features=nom
)
Xtr_bal, ytr_bal = rebalance_training_data(Xtr_a, y_train, target_pos_rate=0.30, seed=SEED)

ann = MLPClassifier(
    hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
    alpha=0.001, batch_size=2048, learning_rate_init=0.001,
    max_iter=60, early_stopping=True, n_iter_no_change=5,
    random_state=SEED,
)
ann.fit(Xtr_bal, ytr_bal)

ann_rv = ann.predict_proba(Xv_a)[:, 1]
ann_rt = ann.predict_proba(Xte_a)[:, 1]
ann_cal = fit_calibrator(ann_rv, y_valid)
ann_ct = apply_calibrator(ann_cal, ann_rt)
ann_auc = roc_auc_score(y_test, ann_ct)
fpr, tpr, _ = roc_curve(y_test, ann_ct)
roc_data['ANN (MLPClassifier)'] = {'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'auc': round(ann_auc, 3)}
print(f'  => ROC-AUC = {ann_auc:.4f}')

## 7. Train PyTorch Tabular Net

In [ ]:
spec = FEATURE_SETS['core_plus_labs_plus_lipids']
feats = [f for f in spec['features'] if f in X_train.columns]
nom = spec['nominal_features']

print(f'Training PyTorch Tabular Net on {len(feats)} features...')

tp = prepare_torch_inputs(
    X_train[feats], X_valid[feats], X_test[feats], nominal_features=nom
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ytr_np = y_train.to_numpy().astype(np.float32)
yv_np  = y_valid.to_numpy().astype(np.float32)

train_ds = TorchTabularDataset(tp['Xtr_n'], tp['Xtr_c'], ytr_np)
valid_ds = TorchTabularDataset(tp['Xv_n'],  tp['Xv_c'],  yv_np)
test_ds  = TorchTabularDataset(tp['Xte_n'], tp['Xte_c'])

train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=4096, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=4096, shuffle=False)

net = TorchTabularNet(
    num_numeric=tp['Xtr_n'].shape[1],
    cat_cards=tp['cards'],
    hidden_dims=(128, 64),
    dropout=0.20,
).to(device)

n_pos = max(float(ytr_np.sum()), 1.0)
n_neg = max(float(len(ytr_np) - ytr_np.sum()), 1.0)
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([n_neg/n_pos], dtype=torch.float32, device=device)
)
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)

best_state, best_ap, patience_ctr = None, -np.inf, 0

for epoch in range(20):
    net.train()
    eloss = 0.0
    for xn, xc, yb in train_loader:
        xn, xc, yb = xn.to(device), xc.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(net(xn, xc), yb)
        loss.backward(); optimizer.step()
        eloss += loss.item() * len(yb)
    tl = eloss / len(train_ds)
    vp = predict_torch_prob(net, valid_loader, device)
    vap = average_precision_score(y_valid, vp)
    vauc = roc_auc_score(y_valid, vp)
    print(f'  Epoch {epoch+1:02d} | train_loss={tl:.4f} | valid_ap={vap:.4f} | valid_auc={vauc:.4f}')
    if vap > best_ap + 1e-6:
        best_ap, best_state, patience_ctr = vap, copy.deepcopy(net.state_dict()), 0
    else:
        patience_ctr += 1
    if patience_ctr >= 4:
        print('  Early stopping.'); break

if best_state: net.load_state_dict(best_state)

torch_rv = predict_torch_prob(net, valid_loader, device)
torch_rt = predict_torch_prob(net, test_loader, device)
torch_cal = fit_calibrator(torch_rv, y_valid)
torch_ct = apply_calibrator(torch_cal, torch_rt)
torch_auc = roc_auc_score(y_test, torch_ct)
fpr, tpr, _ = roc_curve(y_test, torch_ct)
roc_data['Torch Tabular Net'] = {'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'auc': round(torch_auc, 3)}
print(f'  => ROC-AUC = {torch_auc:.4f}')

## 8. Train Logistic Regression, SVM, Decision Tree

In [ ]:
# Prepare sklearn-friendly data
medians = X_train[available].median()
X_train_sk = X_train[available].fillna(medians)
X_test_sk  = X_test[available].fillna(medians)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_sk)
X_test_scaled  = scaler.transform(X_test_sk)

# Logistic Regression
print('Training Logistic Regression...')
lr = LogisticRegression(max_iter=1000, random_state=SEED, class_weight=cw_dict)
lr.fit(X_train_scaled, y_train)
lr_prob = lr.predict_proba(X_test_scaled)[:, 1]
lr_auc = roc_auc_score(y_test, lr_prob)
fpr, tpr, _ = roc_curve(y_test, lr_prob)
roc_data['Logistic Regression'] = {'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'auc': round(lr_auc, 3)}
print(f'  => ROC-AUC = {lr_auc:.4f}')

# SVM (LinearSVC â€” matches original group notebook, trains in seconds on full data)
from sklearn.svm import LinearSVC
print('Training SVM (LinearSVC)...')
svm = LinearSVC(random_state=SEED, class_weight='balanced', max_iter=10000)
svm.fit(X_train_scaled, y_train)
# LinearSVC has no predict_proba â€” use decision_function scores.
# ROC/AUC only depend on ranking, so raw margins are fine.
svm_scores = svm.decision_function(X_test_scaled)
svm_auc = roc_auc_score(y_test, svm_scores)
fpr, tpr, _ = roc_curve(y_test, svm_scores)
roc_data['SVM'] = {'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'auc': round(svm_auc, 3)}
print(f'  => ROC-AUC = {svm_auc:.4f}')

# Decision Tree
print('Training Decision Tree...')
dt = DecisionTreeClassifier(max_depth=10, random_state=SEED, class_weight=cw_dict)
dt.fit(X_train_sk, y_train)
dt_prob = dt.predict_proba(X_test_sk.values)[:, 1]
dt_auc = roc_auc_score(y_test, dt_prob)
fpr, tpr, _ = roc_curve(y_test, dt_prob)
roc_data['Decision Tree'] = {'fpr': fpr.tolist(), 'tpr': tpr.tolist(), 'auc': round(dt_auc, 3)}
print(f'  => ROC-AUC = {dt_auc:.4f}')

## 9. Save ROC Data

In [ ]:
with open('all_roc_data.json', 'w') as f:
    json.dump(roc_data, f)

print('Saved: all_roc_data.json\n')
for name, d in roc_data.items():
    print(f'  {name:<30} AUC = {d["auc"]:.3f}')

## 10. Combined ROC Chart

In [ ]:
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white',
})

fig, ax = plt.subplots(figsize=(8, 7))

colors = {
    'CatBoost (Full Model)':  '#1a5276',
    'CatBoost (Core + Labs)': '#2e86c1',
    'CatBoost (Core Only)':   '#85c1e9',
    'ANN (MLPClassifier)':    '#8e44ad',
    'Torch Tabular Net':      '#d35400',
    'Logistic Regression':    '#e74c3c',
    'SVM':                    '#27ae60',
    'Decision Tree':          '#f39c12',
}
linestyles = {
    'CatBoost (Full Model)':  '-',
    'CatBoost (Core + Labs)': '-',
    'CatBoost (Core Only)':   '-',
    'ANN (MLPClassifier)':    '--',
    'Torch Tabular Net':      '--',
    'Logistic Regression':    '-.',
    'SVM':                    '-.',
    'Decision Tree':          ':',
}

plot_order = [
    'CatBoost (Full Model)',
    'CatBoost (Core + Labs)',
    'Torch Tabular Net',
    'ANN (MLPClassifier)',
    'SVM',
    'Logistic Regression',
    'CatBoost (Core Only)',
    'Decision Tree',
]

for name in plot_order:
    if name not in roc_data: continue
    d = roc_data[name]
    ax.plot(
        d['fpr'], d['tpr'],
        color=colors.get(name, '#333'),
        linestyle=linestyles.get(name, '-'),
        linewidth=2.0,
        label=f'{name} (AUC = {d["auc"]:.3f})',
    )

ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, alpha=0.4)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison \u2014 All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=8.5, framealpha=0.9)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
fig.savefig('roc_all_models.png', dpi=200, bbox_inches='tight')
print('Saved: roc_all_models.png')
plt.show()